# Recomendador Two-Tower sobre MovieLens 20M

Entrena un modelo de **recuperación Two-Tower** (TensorFlow / Keras 3) sobre
[MovieLens 20M](https://files.grouplens.org/datasets/movielens/ml-20m.zip) y exporta los
artefactos que sirve la Lambda `getTwoTowerRecommendations`
(`GET /v1/users/{userId}/profiles/{profileId}/recommendations/ml`).

Pensado para correr entero en **Google Colab gratis** (T4 recomendado; también funciona en CPU,
más lento). No necesita credenciales de AWS: la subida a S3 se hace después, desde el contenedor
`toolbox` del repo de infra.

## Arquitectura

Dos torres que comparten la tabla de embeddings de ítems `E`:

| Torre | Entrada | Salida |
| --- | --- | --- |
| **Item** | índice de película | `E[i] -> Dense(128, relu) -> Dense(64) -> L2` |
| **Query** | historial (hasta 30 películas vistas) | `mean(E[h]) -> Dense(128, relu) -> Dense(64) -> L2` |

La torre de query promedia el historial en vez de usar un embedding por `user_id`. Un embedding
por usuario solo sabría puntuar a los 138k usuarios de MovieLens; promediando el historial el
modelo puntúa a **cualquier** perfil de la app que tenga al menos una película vista, que es lo
que hace que la demo sea interactiva.

Entrenamiento con **softmax muestreado in-batch** (los negativos son los positivos de las otras
filas del lote) más corrección logQ por popularidad.

> No se usa `tensorflow_recommenders`: está clavado a Keras 2 y se rompe con el Keras 3 que trae
> Colab hoy. Las dos torres, la pérdida y las métricas son pocas líneas de Keras puro.

## Evaluación

Split *leave-one-out* cronológico por usuario (última interacción -> test, penúltima -> validación).
Se reportan **recall@K** y **nDCG@K** para K en {10, 20, 50, 100}, rankeando el ítem escondido
contra **todo** el catálogo y enmascarando lo que el usuario ya vio, contra un baseline de
**most-popular** sobre el mismo split.

In [ ]:
import json, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

print("tensorflow", tf.__version__)
print("keras     ", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU       ", gpus if gpus else "sin GPU (va a andar, pero más lento)")

# ── Configuración ────────────────────────────────────────────────────────────────
SEED = 42
MODEL_VERSION = "v1"

MIN_RATING = 4.0     # un rating >= 4 cuenta como interacción positiva
MIN_ITEM_POS = 20    # se descartan películas con menos positivos que esto
MIN_USER_POS = 5     # ... y usuarios con menos positivos que esto

MAX_HIST = 30        # largo máximo del historial que consume la torre de query
PAIRS_PER_USER = 20  # pares (historial, target) muestreados por usuario

DIM = 64             # dimensión del embedding final (la que sirve la Lambda)
HIDDEN = 128
BATCH = 4096
EPOCHS = 30
LR = 1e-3
TEMPERATURE = 0.05   # los vectores están L2-normalizados, así que hay que escalar los logits

KS = [10, 20, 50, 100]
KMAX = max(KS)
EVAL_CHUNK = 2048    # usuarios por bloque al puntuar contra todo el catálogo
VAL_SAMPLE = 10_000  # usuarios usados para el recall@10 de validación por época

DATA_DIR = Path("ml-20m")
ARTIFACTS = Path("artifacts")

keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)

tensorflow 2.20.0
keras      3.13.2
GPU        [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ── 1. Datos ─────────────────────────────────────────────────────────────────────
# ~190 MB. Si la celda se re-ejecuta y el dataset ya está, no vuelve a bajarlo.
if not DATA_DIR.exists():
    if not Path("ml-20m.zip").exists():
        !wget -q --show-progress https://files.grouplens.org/datasets/movielens/ml-20m.zip
    !unzip -q ml-20m.zip

ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(
    DATA_DIR / "movies.csv",
    dtype={"movieId": "int32", "title": "string", "genres": "string"},
)
print(f"{len(ratings):,} ratings, {ratings.userId.nunique():,} usuarios, {len(movies):,} películas")

20,000,263 ratings, 138,493 usuarios, 27,278 películas


In [3]:
# ── 2. Feedback implícito ────────────────────────────────────────────────────────
# El Two-Tower es un modelo de RECUPERACIÓN: no predice la nota, aprende qué ítems son
# relevantes. Un rating >= 4 se toma como positivo y el resto se descarta.
pos = ratings[ratings.rating >= MIN_RATING][["userId", "movieId", "timestamp"]]
print(f"{len(pos):,} positivos (rating >= {MIN_RATING})")

# Un solo pase de filtrado: primero ítems con poca señal, después usuarios cortos.
item_counts = pos.movieId.value_counts()
pos = pos[pos.movieId.isin(item_counts[item_counts >= MIN_ITEM_POS].index)]
user_counts = pos.userId.value_counts()
pos = pos[pos.userId.isin(user_counts[user_counts >= MIN_USER_POS].index)]

# Reindexado: el índice 0 queda reservado como PAD, los ítems reales van de 1..N.
ml_movie_ids = np.sort(pos.movieId.unique())
num_items = len(ml_movie_ids)
pos = pos.assign(
    item=(pd.Categorical(pos.movieId, categories=ml_movie_ids).codes.astype("int32") + 1)
)

# Orden cronológico dentro de cada usuario (mergesort = estable, reproducible con timestamps repetidos).
pos = pos.sort_values(["userId", "timestamp"], kind="mergesort")

items_flat = pos.item.to_numpy(dtype=np.int32)
user_ids = pos.userId.to_numpy()
user_start = np.concatenate([[0], np.flatnonzero(np.diff(user_ids)) + 1]).astype(np.int64)
seq_len = np.diff(np.append(user_start, len(items_flat)))
num_users = len(user_start)

print(f"tras filtrar: {len(items_flat):,} interacciones, {num_users:,} usuarios, {num_items:,} ítems")
print(f"largo de secuencia: mediana {int(np.median(seq_len))}, p95 {int(np.percentile(seq_len, 95))}")

del ratings

9,995,410 positivos (rating >= 4.0)
tras filtrar: 9,936,796 interacciones, 136,659 usuarios, 9,632 ítems
largo de secuencia: mediana 38, p95 249


In [4]:
# ── 3. Split leave-one-out cronológico ───────────────────────────────────────────
# Por usuario: última interacción -> test, penúltima -> validación, el resto -> entrenamiento.
# Es el protocolo estándar para recall@K/nDCG@K en recuperación secuencial, y evita la fuga
# temporal que tendría un split aleatorio.
seqs = np.split(items_flat, user_start[1:])
train_len = seq_len - 2
assert train_len.min() >= 1, "MIN_USER_POS debe garantizar al menos 3 interacciones por usuario"

# Frecuencia de ítem SOLO sobre la parte de entrenamiento: se usa para la corrección logQ
# del softmax in-batch y para el baseline de popularidad.
train_mask = np.ones(len(items_flat), dtype=bool)
train_mask[user_start + seq_len - 1] = False  # test
train_mask[user_start + seq_len - 2] = False  # validación
train_counts = np.bincount(items_flat[train_mask], minlength=num_items + 1).astype(np.float64)

freq = train_counts / max(train_counts[1:].sum(), 1.0)
log_q = np.log(np.maximum(freq, 1e-12)).astype(np.float32)
log_q[0] = 0.0

pop_order = np.argsort(-train_counts[1:], kind="stable")          # filas 0..N-1, de más a menos vista
pop_rank = np.empty(num_items, dtype=np.int64)
pop_rank[pop_order] = np.arange(num_items)
print(f"ítem más visto: {train_counts[1:].max():,.0f} positivos de entrenamiento")

ítem más visto: 54,320 positivos de entrenamiento


In [5]:
# ── 4. Pares (historial, target) de entrenamiento ────────────────────────────────
# Para cada usuario se muestrean hasta PAIRS_PER_USER posiciones t de su tramo de
# entrenamiento; el historial son las MAX_HIST interacciones inmediatamente anteriores,
# rellenadas a la izquierda con el PAD 0.
n_choices = np.clip(np.minimum(train_len - 1, PAIRS_PER_USER), 0, None)
reps = np.repeat(np.arange(num_users), n_choices)
offset_in_user = 1 + np.floor(rng.random(len(reps)) * (train_len[reps] - 1)).astype(np.int64)
target_pos = (user_start[reps] + offset_in_user).astype(np.int64)

train_targets = items_flat[target_pos]
train_hist = np.zeros((len(target_pos), MAX_HIST), dtype=np.int32)

# En bloques: la matriz de posiciones intermedia es del mismo tamaño que el resultado,
# y materializarla entera duplicaría el pico de memoria sin necesidad.
back = np.arange(-MAX_HIST, 0, dtype=np.int64)
BLOCK = 500_000
for s in range(0, len(target_pos), BLOCK):
    e = min(s + BLOCK, len(target_pos))
    hist_pos = target_pos[s:e, None] + back[None, :]
    inside = hist_pos >= user_start[reps[s:e]][:, None]
    train_hist[s:e] = np.where(inside, items_flat[np.clip(hist_pos, 0, None)], 0)

print(f"{len(train_targets):,} pares de entrenamiento, historial promedio "
      f"{(train_hist > 0).sum(axis=1).mean():.1f} ítems")

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_hist, train_targets))
    .shuffle(200_000, seed=SEED)
    .batch(BATCH, drop_remainder=True)   # el softmax in-batch necesita lotes de tamaño fijo
    .prefetch(tf.data.AUTOTUNE)
)

2,374,995 pares de entrenamiento, historial promedio 18.4 ítems


In [6]:
# ── 5. El modelo ─────────────────────────────────────────────────────────────────
class TwoTower(keras.Model):
    """Dos torres que comparten la tabla de embeddings de ítems.

    La torre de item mapea un índice a su vector; la de query promedia el historial
    (enmascarando el PAD) y lo pasa por su propio MLP. Ambas salidas se L2-normalizan,
    así que puntuar es un producto punto — que es exactamente lo que hace la Lambda.
    """

    def __init__(self, num_items, dim, hidden, temperature, log_q, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature
        self.item_table = keras.layers.Embedding(
            num_items + 1, dim, name="item_table",
            embeddings_initializer=keras.initializers.TruncatedNormal(stddev=0.05),
        )
        self.item_d1 = keras.layers.Dense(hidden, activation="relu", name="item_d1")
        self.item_d2 = keras.layers.Dense(dim, name="item_d2")
        self.query_d1 = keras.layers.Dense(hidden, activation="relu", name="query_d1")
        self.query_d2 = keras.layers.Dense(dim, name="query_d2")
        self.log_q = tf.constant(log_q, dtype=tf.float32)
        self.loss_tracker = keras.metrics.Mean(name="loss")

    def encode_items(self, item_idx):
        x = self.item_table(item_idx)
        return tf.math.l2_normalize(self.item_d2(self.item_d1(x)), axis=-1)

    def encode_query(self, history):
        mask = tf.cast(history > 0, tf.float32)
        pooled = tf.reduce_sum(self.item_table(history) * mask[..., None], axis=1)
        pooled = pooled / tf.maximum(tf.reduce_sum(mask, axis=1, keepdims=True), 1.0)
        return tf.math.l2_normalize(self.query_d2(self.query_d1(pooled)), axis=-1)

    def call(self, inputs, training=False):
        history, target = inputs
        return self.encode_query(history), self.encode_items(target)

    def in_batch_loss(self, history, target):
        query = self.encode_query(history)
        candidates = self.encode_items(target)
        logits = tf.matmul(query, candidates, transpose_b=True) / self.temperature

        # Corrección logQ: los negativos in-batch se muestrean con probabilidad proporcional
        # a la popularidad, así que sin esto el modelo aprende a castigar los blockbusters.
        logits -= tf.gather(self.log_q, target)[tf.newaxis, :]

        # "Accidental hits": si el mismo ítem es el target de dos filas del lote, para una de
        # ellas aparece como negativo siendo en realidad positivo. Se anula.
        batch = tf.shape(target)[0]
        duplicated = tf.equal(target[tf.newaxis, :], target[:, tf.newaxis])
        duplicated &= tf.logical_not(tf.eye(batch, dtype=tf.bool))
        logits = tf.where(duplicated, tf.fill(tf.shape(logits), -1e9), logits)

        labels = tf.range(batch)
        return tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        )

    def train_step(self, data):
        history, target = data
        with tf.GradientTape() as tape:
            loss = self.in_batch_loss(history, target)
        self.optimizer.apply_gradients(
            zip(tape.gradient(loss, self.trainable_variables), self.trainable_variables)
        )
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker]


model = TwoTower(num_items, DIM, HIDDEN, TEMPERATURE, log_q)
model.compile(optimizer=keras.optimizers.Adam(LR))

In [7]:
# ── 6. Evaluación: recall@K y nDCG@K ─────────────────────────────────────────────
def build_eval_arrays(split):
    """split='val' -> target = penúltima interacción; split='test' -> última.

    El historial es todo lo anterior al target (recortado a MAX_HIST) y también es
    exactamente lo que hay que enmascarar al rankear: recomendar algo ya visto no cuenta.
    """
    cut = -2 if split == "val" else -1
    hist, targets, seen = [], [], []
    for s in seqs:
        base = s[:cut]
        if len(base) == 0:
            continue
        window = base[-MAX_HIST:]
        row = np.zeros(MAX_HIST, dtype=np.int32)
        row[MAX_HIST - len(window):] = window
        hist.append(row)
        targets.append(s[cut])
        seen.append(base)
    return np.stack(hist), np.asarray(targets, dtype=np.int32), seen


def encode_queries(model, hist, batch=8192):
    out = np.empty((len(hist), DIM), dtype=np.float32)
    for s in range(0, len(hist), batch):
        out[s:s + batch] = model.encode_query(hist[s:s + batch]).numpy()
    return out


def compute_item_vectors(model, batch=8192):
    out = np.empty((num_items, DIM), dtype=np.float32)
    for s in range(0, num_items, batch):
        idx = np.arange(s + 1, min(s + batch, num_items) + 1, dtype=np.int32)
        out[s:s + len(idx)] = model.encode_items(idx).numpy()
    return out


def target_ranks(query_vecs, item_vecs, targets, seen, kmax):
    """Posición 0-based del target en el ranking completo, o -1 si cae fuera del top-kmax."""
    ranks = np.full(len(targets), -1, dtype=np.int64)
    for s in range(0, len(targets), EVAL_CHUNK):
        e = min(s + EVAL_CHUNK, len(targets))
        scores = query_vecs[s:e] @ item_vecs.T          # (bloque, num_items)
        for i in range(s, e):
            scores[i - s, seen[i] - 1] = -np.inf        # índice de modelo 1..N -> fila 0..N-1
        part = np.argpartition(-scores, kmax, axis=1)[:, :kmax]
        ordered = np.take_along_axis(
            part, np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1), axis=1
        )
        hit = ordered == (targets[s:e, None] - 1)
        ranks[s:e] = np.where(hit.any(axis=1), hit.argmax(axis=1), -1)
    return ranks


def metrics_from_ranks(ranks, ks=KS):
    out = {}
    for k in ks:
        found = (ranks >= 0) & (ranks < k)
        gains = np.where(found, 1.0 / np.log2(np.maximum(ranks, 0) + 2.0), 0.0)
        out[f"recall@{k}"] = float(found.mean())
        out[f"ndcg@{k}"] = float(gains.mean())
    return out


def popularity_ranks(targets, seen, kmax):
    """Baseline most-popular. El ranking es fijo, así que la posición del target es su
    posición global menos cuántos ítems más populares fueron enmascarados por 'ya visto'."""
    ranks = np.empty(len(targets), dtype=np.int64)
    for i, (t, s) in enumerate(zip(targets, seen)):
        p = pop_rank[t - 1]
        ranks[i] = p - int((pop_rank[s - 1] < p).sum())
    return np.where(ranks < kmax, ranks, -1)


val_hist, val_targets, val_seen = build_eval_arrays("val")
test_hist, test_targets, test_seen = build_eval_arrays("test")
print(f"validación: {len(val_targets):,} usuarios · test: {len(test_targets):,} usuarios")

# Submuestra fija para el seguimiento por época (evaluar los 130k usuarios cada época es caro).
val_sample = rng.choice(len(val_targets), size=min(VAL_SAMPLE, len(val_targets)), replace=False)

validación: 136,659 usuarios · test: 136,659 usuarios


In [8]:
# ── 7. Entrenamiento ─────────────────────────────────────────────────────────────
class ValRecall(keras.callbacks.Callback):
    """recall@10 de validación al final de cada época, y restauración de los mejores pesos.

    No se usa EarlyStopping porque la métrica que importa (recall sobre todo el catálogo) no
    es una métrica de Keras: hay que recalcular los vectores de ítem en cada época.
    """

    def __init__(self, k=10):
        super().__init__()
        self.k = k
        self.best = -1.0
        self.best_weights = None
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        item_vecs = compute_item_vectors(self.model)
        query_vecs = encode_queries(self.model, val_hist[val_sample])
        ranks = target_ranks(
            query_vecs, item_vecs, val_targets[val_sample],
            [val_seen[i] for i in val_sample], self.k,
        )
        recall = float(((ranks >= 0) & (ranks < self.k)).mean())
        self.history.append(recall)
        (logs or {})[f"val_recall@{self.k}"] = recall
        marker = ""
        if recall > self.best:
            self.best, self.best_weights, marker = recall, self.model.get_weights(), "  <- mejor"
        print(f"    val recall@{self.k} = {recall:.4f}{marker}")


val_recall = ValRecall()
started = time.time()
model.fit(train_ds, epochs=EPOCHS, callbacks=[val_recall], verbose=1)
print(f"\nentrenamiento: {time.time() - started:.0f}s")

if val_recall.best_weights is not None:
    model.set_weights(val_recall.best_weights)
    print(f"pesos restaurados de la mejor época (val recall@10 = {val_recall.best:.4f})")

Epoch 1/30
578/579 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 7.9712    val recall@10 = 0.1406  <- mejor
579/579 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - loss: 7.4292 - val_recall@10: 0.1406
Epoch 2/30
579/579 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 6.9805    val recall@10 = 0.1521  <- mejor
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - loss: 6.9302 - val_recall@10: 0.1521
Epoch 3/30
579/579 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 6.8464    val recall@10 = 0.1575  <- mejor
579/579 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - loss: 6.8196 - val_recall@10: 0.1575
Epoch 4/30
579/579 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 6.7789    val recall@10 = 0.1627  <- mejor
579/579 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - loss: 6.7583 - val_recall@10: 0.1627
Epoch 5/30
578/579 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 6.7316    val recall@10 = 0.1616
579/579 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - loss: 6.7150 - val_recall@10: 0.1616
Epoch 6/30
578/579 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 6.6991    val recal

In [9]:
# ── 8. Métricas finales sobre el conjunto de test ────────────────────────────────
item_vecs = compute_item_vectors(model)
test_query_vecs = encode_queries(model, test_hist)

model_metrics = metrics_from_ranks(target_ranks(test_query_vecs, item_vecs, test_targets, test_seen, KMAX))
baseline_metrics = metrics_from_ranks(popularity_ranks(test_targets, test_seen, KMAX))

header = f"{'métrica':<12}{'two-tower':>12}{'most-popular':>15}{'lift':>10}"
print(header)
print("-" * len(header))
for name in [f"recall@{k}" for k in KS] + [f"ndcg@{k}" for k in KS]:
    m, b = model_metrics[name], baseline_metrics[name]
    lift = f"{m / b:.1f}x" if b > 0 else "n/a"
    print(f"{name:<12}{m:>12.4f}{b:>15.4f}{lift:>10}")

metrics = {
    "modelVersion": MODEL_VERSION,
    "dataset": "movielens-20m",
    "evalProtocol": "leave-one-out cronológico, ranking contra el catálogo completo, ítems vistos enmascarados",
    "evalUsers": int(len(test_targets)),
    "numItems": int(num_items),
    "model": model_metrics,
    "popularityBaseline": baseline_metrics,
    "config": {
        "minRating": MIN_RATING, "minItemPositives": MIN_ITEM_POS, "minUserPositives": MIN_USER_POS,
        "maxHistory": MAX_HIST, "pairsPerUser": PAIRS_PER_USER, "dim": DIM, "hidden": HIDDEN,
        "batch": BATCH, "epochs": EPOCHS, "lr": LR, "temperature": TEMPERATURE, "seed": SEED,
    },
}

métrica        two-tower   most-popular      lift
-------------------------------------------------
recall@10         0.1470         0.0511      2.9x
recall@20         0.2207         0.0844      2.6x
recall@50         0.3507         0.1453      2.4x
recall@100        0.4759         0.2200      2.2x
ndcg@10           0.0786         0.0259      3.0x
ndcg@20           0.0972         0.0343      2.8x
ndcg@50           0.1228         0.0463      2.7x
ndcg@100          0.1431         0.0583      2.5x


In [10]:
# ── 9. Exportar los artefactos que sirve la Lambda ───────────────────────────────
# Todas las matrices quedan 0-based sobre los N ítems reales: la fila i corresponde al
# índice i de catalog.json (la fila PAD del modelo se descarta acá).
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# float32 little-endian, row-major: la Lambda lo lee como Float32Array sin parsear nada.
item_vecs.astype("<f4").tofile(ARTIFACTS / "item_vectors.f32")
model.item_table.embeddings.numpy()[1:].astype("<f4").tofile(ARTIFACTS / "pool_embeddings.f32")

def weights(dense):
    return np.round(dense.kernel.numpy(), 6).tolist(), np.round(dense.bias.numpy(), 6).tolist()

w1, b1 = weights(model.query_d1)
w2, b2 = weights(model.query_d2)
(ARTIFACTS / "query_tower.json").write_text(json.dumps({
    "dim": DIM, "hidden": HIDDEN, "activation": "relu",
    "w1": w1, "b1": b1, "w2": w2, "b2": b2,
}))

# ── catalog.json: el puente entre el índice del modelo y el catálogo de la app ──
# La Lambda cruza esto contra MoviesTable por (source, externalId) o por título+año.
TITLE_YEAR = re.compile(r"^(?P<name>.*?)\s*\((?P<year>\d{4})\)\s*$")
TRAILING_ARTICLE = re.compile(r"^(?P<rest>.+),\s*(?P<article>The|A|An|Le|La|Les|L'|El|Los|Las|Il|Der|Die|Das|Den|Det|De)$", re.IGNORECASE)

def clean_title(raw):
    """'Matrix, The (1999)' -> ('The Matrix', 1999). MovieLens pospone el artículo."""
    raw = str(raw).strip()
    match = TITLE_YEAR.match(raw)
    name, year = (match.group("name"), int(match.group("year"))) if match else (raw, None)
    article = TRAILING_ARTICLE.match(name)
    if article:
        art, rest = article.group("article"), article.group("rest")
        name = f"{art}{rest}" if art.endswith("'") else f"{art} {rest}"
    return name.strip(), year

meta = movies.set_index("movieId")
catalog_items = []
for row, ml_id in enumerate(ml_movie_ids):
    title_raw = meta.title.get(ml_id, str(ml_id))
    genres_raw = meta.genres.get(ml_id, "")
    title, year = clean_title(title_raw)
    genres = [g for g in str(genres_raw).split("|") if g and g != "(no genres listed)"]
    catalog_items.append({
        "mlMovieId": int(ml_id),
        "title": title,
        "year": year,
        "genres": genres,
        "popRank": int(pop_rank[row]),
    })

(ARTIFACTS / "catalog.json").write_text(json.dumps({
    "modelVersion": MODEL_VERSION, "dim": DIM, "count": num_items, "items": catalog_items,
}))
(ARTIFACTS / "metrics.json").write_text(json.dumps(metrics, indent=2))

for f in sorted(ARTIFACTS.iterdir()):
    print(f"{f.name:<24}{f.stat().st_size / 1e6:>8.2f} MB")

# Chequeo de integridad: la Lambda lee estos archivos asumiendo exactamente estas formas.
assert (ARTIFACTS / "item_vectors.f32").stat().st_size == num_items * DIM * 4
assert (ARTIFACTS / "pool_embeddings.f32").stat().st_size == num_items * DIM * 4
assert np.allclose(np.linalg.norm(item_vecs, axis=1), 1.0, atol=1e-4), "item_vectors debe estar L2-normalizado"
print("\nOK: formas y normas verificadas")

catalog.json                1.15 MB
item_vectors.f32            2.47 MB
metrics.json                0.00 MB
pool_embeddings.f32         2.47 MB
query_tower.json            0.36 MB

OK: formas y normas verificadas


In [ ]:
# ── 10. Descargar ────────────────────────────────────────────────────────────────
shutil.make_archive("two_tower_artifacts", "zip", ARTIFACTS)
print(f"two_tower_artifacts.zip -> {Path('two_tower_artifacts.zip').stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files
    files.download("two_tower_artifacts.zip")
except ImportError:
    print("No estás en Colab: copiá two_tower_artifacts.zip a mano.")

## Subir los artefactos a S3

Descomprimir el zip en `ml/artifacts/` del repo de infra (ya está en `.gitignore`: son blobs de
varios MB, viven en S3 y no en git) y subirlos desde el contenedor `toolbox`, que ya trae AWS CLI v2:

```bash
docker compose run --rm toolbox bash

# El bucket lo nombra CloudFormation, así que se lee del output del stack — el mismo
# mecanismo que usa scripts/create-admin-user.sh para el CognitoUserPoolId.
BUCKET=$(aws cloudformation describe-stacks \
  --stack-name ProyectoNetflixInfraStack \
  --query "Stacks[0].Outputs[?OutputKey=='ModelArtifactsBucketName'].OutputValue" \
  --output text)

aws s3 sync ml/artifacts "s3://$BUCKET/two-tower/v3/"
```

El prefijo `two-tower/v3/` tiene que coincidir con `MODEL_ARTIFACT_PREFIX` en el stack
(`lib/proyecto_netflix-infra-stack.ts`). Para publicar un modelo nuevo sin tocar el que está
sirviendo: subir a `two-tower/v4/`, cambiar la variable de entorno y desplegar — el rollback es
volver a apuntar el prefijo.

La Lambda cachea los artefactos por contenedor, así que después de subir una versión nueva las
Lambdas tibias siguen sirviendo la anterior hasta que se reciclan.